# Deduplication Notebook

This notebook deletes duplicated data permanently from the cleaned datasets and updates the pretreatment pipeline.

In [6]:
import pandas as pd
import json
from pathlib import Path

DATA_DIR = Path('.')
CLEANED_DIR = DATA_DIR / 'cleaned'

def deduplicate_file(name):
    csv_path = CLEANED_DIR / f'{name}.csv'
    json_path = CLEANED_DIR / f'{name}.json'
    
    if not csv_path.exists():
        return
        
    print(f'-- Processing {name} --')
    df = pd.read_csv(csv_path, low_memory=False)
    original_count = len(df)
    
    if 'is_duplicate' in df.columns:
        # Drop duplicates based on the boolean column
        df_clean = df[~df['is_duplicate']].copy()
        # Also drop exact ID duplicates if any slipped through
        df_clean = df_clean.drop_duplicates(subset=['id'])
        
        removed = original_count - len(df_clean)
        print(f'Removed {removed:,} duplicates from {original_count:,} total.')
        
        if removed > 0:
            df_clean.to_csv(csv_path, index=False, encoding='utf-8-sig')
            df_clean.to_json(json_path, orient='records', force_ascii=False, indent=2)
            print('Saved cleaned CSV and JSON files.\n')
        else:
            print('No duplicates found.\n')
    else:
        print('No is_duplicate column.\n')


In [8]:
datasets = ['unified_dataset']

for ds in datasets:
    deduplicate_file(ds)


-- Processing unified_dataset --
Removed 3,136 duplicates from 37,492 total.
Saved cleaned CSV and JSON files.



## Update `pretreatment.py`

This cell updates `pretreatment.py` so that future runs of the data cleaning pipeline automatically drop duplicates instead of just flagging them with `True`.

In [4]:
pretreatment_path = DATA_DIR / 'pretreatment.py'

if pretreatment_path.exists():
    content = pretreatment_path.read_text(encoding='utf-8')
    
    old_code = '# 5. Mark duplicates\n    df = flag_duplicates(df)'
    new_code = '# 5. Mark duplicates and drop them\n    df = flag_duplicates(df)\n    dup_count = df["is_duplicate"].sum()\n    df = df[~df["is_duplicate"]].copy()'
    
    if old_code in content:
        content = content.replace(old_code, new_code)
        
        old_qa = 'dup_count     = df["is_duplicate"].sum()'
        new_qa = '# dup_count computed above'
        content = content.replace(old_qa, new_qa)
        
        pretreatment_path.write_text(content, encoding='utf-8')
        print('Updated pretreatment.py to drop duplicates automatically on future runs.')
    else:
        print('pretreatment.py already updated or exact code not found.')
else:
    print('pretreatment.py not found in the directory.')


Updated pretreatment.py to drop duplicates automatically on future runs.
